# backward-func-lookup — faded example 1: Complete add_back_func's tuple-keyed insert

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `backward-func-lookup`. The last cell reports your progress on the `Backprop: BackwardFuncLookup` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: BackwardFuncLookup` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-func-lookup`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-func-lookup"
DD_SUBTOPIC = "Backprop: BackwardFuncLookup"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`BackwardFuncLookup` stores back fns in a flat dict keyed by `(forward_fn, arg_position)`. The whole job of `add_back_func` is a single dict assignment using that 2-tuple key; overwriting an existing key is intentional so re-registration is cheap.

## Faded exercise 1

Complete `add_back_func` so it registers `back_fn` under the `(forward_fn, arg_position)` key. The `get_back_func` method and the test harness are already written; you only fill the one insert line.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}

    def add_back_func(self, forward_fn, arg_position, back_fn):
        self.back_funcs[(forward_fn, arg_position)] = back_fn

    def get_back_func(self, forward_fn, arg_position):
        key = (forward_fn, arg_position)
        if key not in self.back_funcs:
            raise KeyError(f'No back_fn for ({forward_fn!r}, argnum={arg_position}).')
        return self.back_funcs[key]


def _test():
    def fn_a(*a):
        return a
    def fn_b(*a):
        return a
    BFL = BackwardFuncLookup()
    BFL.add_back_func(t.log, 0, fn_a)
    BFL.add_back_func(t.multiply, 1, fn_b)
    # stored under the exact tuple keys
    assert BFL.back_funcs[(t.log, 0)] is fn_a
    assert BFL.back_funcs[(t.multiply, 1)] is fn_b
    # get returns the same object
    assert BFL.get_back_func(t.log, 0) is fn_a
    assert BFL.get_back_func(t.multiply, 1) is fn_b
    # overwrite does not grow the dict and replaces the value
    BFL.add_back_func(t.log, 0, fn_b)
    assert len(BFL.back_funcs) == 2
    assert BFL.get_back_func(t.log, 0) is fn_b
    # missing key raises KeyError
    try:
        BFL.get_back_func(t.exp, 0)
        raised = False
    except KeyError:
        raised = True
    assert raised


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}

    def add_back_func(self, forward_fn, arg_position, back_fn):
        self.back_funcs[(forward_fn, arg_position)] = back_fn

    def get_back_func(self, forward_fn, arg_position):
        key = (forward_fn, arg_position)
        if key not in self.back_funcs:
            raise KeyError(f'No back_fn for ({forward_fn!r}, argnum={arg_position}).')
        return self.back_funcs[key]
```
</details>